In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
# ============================================================
# CONFIGURATION
# ============================================================

# Define the ADLS paths containing the Gold analytical datasets
# and the output dataset generated by anomaly detection.
account = "mastermmf001sta"

lakehouse_path = (
    f"abfss://lakehouse@{account}.dfs.core.windows.net"
)

gold_daily_path = (
    f"{lakehouse_path}/gold/daily_production"
)

gold_spatial_path = (
    f"{lakehouse_path}/gold/spatial_monitoring"
)

gold_anomalies_path = (
    f"{lakehouse_path}/gold/detected_anomalies"
)

spark.conf.set(
  "fs.azure.account.key.mastermmf001sta.dfs.core.windows.net",
  "Mkp5+QxUJpJ5/r64MPHfZruiqhiXr+jGHqV3yfUynm2pZG7KiY8t3k6lp9JHfI3YB78B4pPNr1er+AStK6ZnnA=="
)

In [0]:
# ============================================================
# READ GOLD DATASETS
# ============================================================

# Load the daily and spatial analytical datasets generated
# in the Gold analytics stage.
df_daily = (
    spark.read
    .format("delta")
    .load(gold_daily_path)
)

df_spatial = (
    spark.read
    .format("delta")
    .load(gold_spatial_path)
)

print(f"Daily Gold rows: {df_daily.count()}")
print(f"Spatial Gold rows: {df_spatial.count()}")

Daily Gold rows: 105
Spatial Gold rows: 4860


In [0]:
# ============================================================
# CALCULATE ROBUST TEMPORAL BASELINE
# ============================================================

# Select the variables required for temporal anomaly detection.
# Each observation represents the production of one battery on one day.
df_temporal_base = (
    df_daily
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "capture_date",
        "mean_egg_count"
    )
)

# Create two aliases of the dataset.
# The first represents the day being evaluated and the second
# provides the historical days used to calculate its baseline.
current = df_temporal_base.alias("current")
reference = df_temporal_base.alias("reference")

# Match each battery-day observation with all the OTHER days
# available for the same battery.
df_temporal_pairs = (
    current
    .join(
        reference,
        (
            (F.col("current.farm_id") == F.col("reference.farm_id"))
            &
            (
                F.col("current.house_number")
                == F.col("reference.house_number")
            )
            &
            (
                F.col("current.battery_number")
                == F.col("reference.battery_number")
            )
            &
            (
                F.col("current.capture_date")
                != F.col("reference.capture_date")
            )
        ),
        "inner"
    )
)

In [0]:
# ============================================================
# CALCULATE MEDIAN BASELINE
# ============================================================

# Calculate the median production of the other available days.
# The median is used because it is less sensitive to extreme
# observations than the arithmetic mean.
df_temporal = (
    df_temporal_pairs
    .groupBy(
        F.col("current.farm_id").alias("farm_id"),
        F.col("current.house_number").alias("house_number"),
        F.col("current.battery_number").alias("battery_number"),
        F.col("current.capture_date").alias("capture_date"),
        F.col("current.mean_egg_count").alias("mean_egg_count")
    )
    .agg(
        F.percentile_approx(
            F.col("reference.mean_egg_count"),
            0.5
        ).alias("baseline_median")
    )
)

In [0]:
# ============================================================
# CALCULATE TEMPORAL DEVIATION
# ============================================================

# Calculate how much the observed production differs from
# the robust baseline of the same battery.
df_temporal = (
    df_temporal
    .withColumn(
        "deviation_pct",
        F.round(
            (
                F.col("mean_egg_count")
                - F.col("baseline_median")
            )
            / F.col("baseline_median")
            * 100,
            2
        )
    )
)

In [0]:
# ============================================================
# REVIEW TEMPORAL DEVIATIONS
# ============================================================

# Display battery-day observations ordered from the largest
# production decrease to the largest production increase.
display(
    df_temporal
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "capture_date",
        "mean_egg_count",
        F.round(
            "baseline_median",
            3
        ).alias("baseline_median"),
        "deviation_pct"
    )
    .orderBy("deviation_pct")
)

farm_id,house_number,battery_number,capture_date,mean_egg_count,baseline_median,deviation_pct
farm_03,1,3,2026-08-04,0.507,0.993,-48.94
farm_03,2,1,2026-08-04,0.509,0.975,-47.79
farm_03,1,1,2026-08-04,0.515,0.982,-47.56
farm_03,2,3,2026-08-04,0.514,0.975,-47.28
farm_03,1,4,2026-08-04,0.525,0.989,-46.92
farm_03,2,2,2026-08-04,0.535,0.98,-45.41
farm_03,1,2,2026-08-04,0.53,0.969,-45.3
farm_01,1,1,2026-08-02,0.932,0.992,-6.05
farm_01,2,3,2026-08-02,0.955,0.998,-4.31
farm_03,1,4,2026-08-02,0.948,0.989,-4.15


In [0]:
# ============================================================
# CALCULATE TEMPORAL ANOMALY THRESHOLD
# ============================================================

# Calculate the first and third quartiles of the temporal
# deviation distribution.
quartiles = (
    df_temporal
    .approxQuantile(
        "deviation_pct",
        [0.25, 0.75],
        0.0
    )
)

q1 = quartiles[0]
q3 = quartiles[1]

# Calculate the interquartile range (IQR).
iqr = q3 - q1

# Define the lower threshold used to identify unusually
# large decreases in production.
lower_threshold = q1 - 1.5 * iqr

print(f"Q1: {q1:.2f} %")
print(f"Q3: {q3:.2f} %")
print(f"IQR: {iqr:.2f} %")
print(f"Lower anomaly threshold: {lower_threshold:.2f} %")

Q1: -1.54 %
Q3: 2.85 %
IQR: 4.39 %
Lower anomaly threshold: -8.12 %


In [0]:
# ============================================================
# DETECT TEMPORAL ANOMALIES
# ============================================================

# Flag observations whose production deviation is below
# the automatically calculated IQR lower threshold.
df_temporal_detected = (
    df_temporal
    .withColumn(
        "is_anomaly",
        F.col("deviation_pct") < F.lit(lower_threshold)
    )
)

In [0]:
# ============================================================
# REVIEW DETECTED TEMPORAL ANOMALIES
# ============================================================

# Display only observations automatically classified
# as temporal production anomalies.
display(
    df_temporal_detected
    .filter(F.col("is_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "capture_date",
        "mean_egg_count",
        F.round(
            "baseline_median",
            3
        ).alias("baseline_median"),
        "deviation_pct",
        "is_anomaly"
    )
    .orderBy("deviation_pct")
)

farm_id,house_number,battery_number,capture_date,mean_egg_count,baseline_median,deviation_pct,is_anomaly
farm_03,1,3,2026-08-04,0.507,0.993,-48.94,true
farm_03,2,1,2026-08-04,0.509,0.975,-47.79,true
farm_03,1,1,2026-08-04,0.515,0.982,-47.56,true
farm_03,2,3,2026-08-04,0.514,0.975,-47.28,true
farm_03,1,4,2026-08-04,0.525,0.989,-46.92,true
farm_03,2,2,2026-08-04,0.535,0.98,-45.41,true
farm_03,1,2,2026-08-04,0.53,0.969,-45.3,true


In [0]:
# ============================================================
# TEMPORAL ANOMALY DETECTION SUMMARY
# ============================================================

# Summarize the number of observations evaluated and
# the number automatically classified as temporal anomalies.
temporal_summary = (
    df_temporal_detected
    .agg(
        F.count("*").alias("evaluated_observations"),
        F.sum(
            F.col("is_anomaly").cast("int")
        ).alias("detected_anomalies")
    )
)

display(temporal_summary)

evaluated_observations,detected_anomalies
105,7


In [0]:
# ============================================================
# PREPARE SPATIAL ANOMALY DETECTION
# ============================================================

# Select the variables required to detect abnormal spatial
# behaviour within each battery and recording date.
df_spatial_base = (
    df_spatial
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "capture_date",
        "mean_egg_count",
        "zero_egg_rate_pct",
        "error_rate_pct"
    )
)

In [0]:
# ============================================================
# CALCULATE SPATIAL BASELINE
# ============================================================

# Define a window containing all spatial groups belonging
# to the same battery and recording date.
spatial_window = (
    Window
    .partitionBy(
        "farm_id",
        "house_number",
        "battery_number",
        "capture_date"
    )
)

# Calculate the median spatial behaviour of each battery.
# The median provides a robust reference against extreme zones.
df_spatial_detect = (
    df_spatial_base
    .withColumn(
        "battery_median_egg_count",
        F.expr(
            """
            percentile_approx(
                mean_egg_count,
                0.5
            )
            """
        ).over(spatial_window)
    )
    .withColumn(
        "battery_median_zero_rate",
        F.expr(
            """
            percentile_approx(
                zero_egg_rate_pct,
                0.5
            )
            """
        ).over(spatial_window)
    )
)

In [0]:
# ============================================================
# CALCULATE SPATIAL DEVIATION
# ============================================================

# Measure how much the mean egg count of each spatial group
# differs from the median behaviour of its own battery.
df_spatial_detect = (
    df_spatial_detect
    .withColumn(
        "spatial_deviation_pct",
        F.round(
            (
                F.col("mean_egg_count")
                - F.col("battery_median_egg_count")
            )
            / F.col("battery_median_egg_count")
            * 100,
            2
        )
    )
)

In [0]:
# ============================================================
# REVIEW LARGEST SPATIAL DEVIATIONS
# ============================================================

# Display spatial groups ordered by the largest negative
# deviation from the typical behaviour of their battery.
display(
    df_spatial_detect
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "capture_date",
        "mean_egg_count",
        F.round(
            "battery_median_egg_count",
            3
        ).alias("battery_median_egg_count"),
        "zero_egg_rate_pct",
        "spatial_deviation_pct"
    )
    .orderBy("spatial_deviation_pct")
)

farm_id,house_number,battery_number,level,side,position_group,capture_date,mean_egg_count,battery_median_egg_count,zero_egg_rate_pct,spatial_deviation_pct
farm_02,2,4,1,FRONT,451,2026-08-01,0.0,0.98,100.0,-100.0
farm_02,2,4,2,BACK,451,2026-08-01,0.0,0.98,100.0,-100.0
farm_02,2,4,2,FRONT,451,2026-08-01,0.0,0.98,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-01,0.0,0.98,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-02,0.0,0.94,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-02,0.0,0.94,100.0,-100.0
farm_02,2,4,2,FRONT,451,2026-08-02,0.0,0.94,100.0,-100.0
farm_02,2,4,2,BACK,451,2026-08-02,0.0,0.94,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-03,0.0,1.0,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-03,0.0,1.0,100.0,-100.0


In [0]:
# ============================================================
# SUMMARIZE SPATIAL BEHAVIOUR ACROSS DAYS
# ============================================================

# Aggregate the spatial deviation of each physical zone across
# all available dates to identify persistent spatial patterns.
df_spatial_persistent = (
    df_spatial_detect
    .groupBy(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group"
    )
    .agg(
        # Number of days available for the spatial group.
        F.countDistinct("capture_date").alias("number_of_days"),

        # Median deviation across days provides a robust
        # measure of the typical behaviour of the zone.
        F.expr(
            "percentile_approx(spatial_deviation_pct, 0.5)"
        ).alias("median_spatial_deviation_pct"),

        # Calculate the typical zero-egg rate of the zone.
        F.expr(
            "percentile_approx(zero_egg_rate_pct, 0.5)"
        ).alias("median_zero_egg_rate_pct")
    )
)

In [0]:
# ============================================================
# REVIEW PERSISTENT SPATIAL DEVIATIONS
# ============================================================

# Display the zones with the largest persistent negative
# deviations across the available recording days.
display(
    df_spatial_persistent
    .orderBy("median_spatial_deviation_pct")
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,median_spatial_deviation_pct,median_zero_egg_rate_pct
farm_02,2,4,1,BACK,451,5,-100.0,100.0
farm_02,2,4,1,FRONT,451,5,-100.0,100.0
farm_02,2,4,2,BACK,451,5,-100.0,100.0
farm_02,2,4,2,FRONT,451,5,-100.0,100.0
farm_03,1,4,3,FRONT,301,5,-18.67,24.0
farm_01,2,2,1,BACK,351,5,-11.76,16.0
farm_03,2,2,3,FRONT,251,5,-11.11,18.0
farm_02,1,1,1,BACK,151,5,-10.0,12.0
farm_03,1,4,4,FRONT,401,5,-8.45,14.29
farm_02,2,3,3,BACK,201,5,-8.33,14.0


In [0]:
# ============================================================
# CALCULATE SPATIAL ANOMALY THRESHOLD
# ============================================================

# Calculate the first and third quartiles of the persistent
# spatial deviation distribution.
quartiles = (
    df_spatial_persistent
    .approxQuantile(
        "median_spatial_deviation_pct",
        [0.25, 0.75],
        0.0
    )
)

q1_spatial = quartiles[0]
q3_spatial = quartiles[1]

# Calculate the interquartile range (IQR).
iqr_spatial = q3_spatial - q1_spatial

# Define the lower threshold used to identify spatial zones
# with unusually low production.
lower_spatial_threshold = (
    q1_spatial - 1.5 * iqr_spatial
)

print(f"Q1: {q1_spatial:.2f} %")
print(f"Q3: {q3_spatial:.2f} %")
print(f"IQR: {iqr_spatial:.2f} %")
print(
    f"Lower spatial anomaly threshold: "
    f"{lower_spatial_threshold:.2f} %"
)

Q1: -2.00 %
Q3: 2.04 %
IQR: 4.04 %
Lower spatial anomaly threshold: -8.06 %


In [0]:
# ============================================================
# EVALUATE SPATIAL IQR THRESHOLD
# ============================================================

# Apply the IQR threshold temporarily to evaluate how many
# spatial groups would be classified as anomalous.
df_spatial_iqr_test = (
    df_spatial_persistent
    .withColumn(
        "is_iqr_anomaly",
        F.col("median_spatial_deviation_pct")
        < F.lit(lower_spatial_threshold)
    )
)

# Count the total number of spatial groups evaluated and
# the number that would be classified as anomalies.
spatial_iqr_summary = (
    df_spatial_iqr_test
    .agg(
        F.count("*").alias("evaluated_spatial_groups"),
        F.sum(
            F.col("is_iqr_anomaly").cast("int")
        ).alias("iqr_anomalies")
    )
)

display(spatial_iqr_summary)

evaluated_spatial_groups,iqr_anomalies
972,15


In [0]:
# ============================================================
# REVIEW SPATIAL IQR CANDIDATES
# ============================================================

# Display all spatial groups that fall below the IQR threshold
# to assess whether the statistical criterion is sufficiently
# selective for persistent spatial anomaly detection.
display(
    df_spatial_iqr_test
    .filter(F.col("is_iqr_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "number_of_days",
        "median_spatial_deviation_pct",
        "median_zero_egg_rate_pct"
    )
    .orderBy("median_spatial_deviation_pct")
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,median_spatial_deviation_pct,median_zero_egg_rate_pct
farm_02,2,4,1,BACK,451,5,-100.0,100.0
farm_02,2,4,1,FRONT,451,5,-100.0,100.0
farm_02,2,4,2,BACK,451,5,-100.0,100.0
farm_02,2,4,2,FRONT,451,5,-100.0,100.0
farm_03,1,4,3,FRONT,301,5,-18.67,24.0
farm_01,2,2,1,BACK,351,5,-11.76,16.0
farm_03,2,2,3,FRONT,251,5,-11.11,18.0
farm_02,1,1,1,BACK,151,5,-10.0,12.0
farm_03,1,4,4,FRONT,401,5,-8.45,14.29
farm_02,2,3,3,BACK,201,5,-8.33,14.0


In [0]:
# ============================================================
# CALCULATE ZERO-EGG RATE ANOMALY THRESHOLD
# ============================================================

# Calculate the first and third quartiles of the median
# zero-egg rate across persistent spatial groups.
zero_rate_quartiles = (
    df_spatial_persistent
    .approxQuantile(
        "median_zero_egg_rate_pct",
        [0.25, 0.75],
        0.0
    )
)

q1_zero = zero_rate_quartiles[0]
q3_zero = zero_rate_quartiles[1]

# Calculate the interquartile range (IQR).
iqr_zero = q3_zero - q1_zero

# Define the upper threshold because spatial problems are
# represented by unusually high percentages of zero counts.
upper_zero_threshold = (
    q3_zero + 1.5 * iqr_zero
)

print(f"Q1: {q1_zero:.2f} %")
print(f"Q3: {q3_zero:.2f} %")
print(f"IQR: {iqr_zero:.2f} %")
print(
    f"Upper zero-egg anomaly threshold: "
    f"{upper_zero_threshold:.2f} %"
)

Q1: 8.00 %
Q3: 12.00 %
IQR: 4.00 %
Upper zero-egg anomaly threshold: 18.00 %


In [0]:
# ============================================================
# EVALUATE ZERO-EGG RATE THRESHOLD
# ============================================================

# Apply the automatically calculated IQR threshold to identify
# spatial groups with an unusually high persistent zero-egg rate.
df_spatial_zero_test = (
    df_spatial_persistent
    .withColumn(
        "is_zero_rate_anomaly",
        F.col("median_zero_egg_rate_pct")
        > F.lit(upper_zero_threshold)
    )
)

# Count how many spatial groups would be classified
# as anomalous using this criterion.
spatial_zero_summary = (
    df_spatial_zero_test
    .agg(
        F.count("*").alias("evaluated_spatial_groups"),
        F.sum(
            F.col("is_zero_rate_anomaly").cast("int")
        ).alias("detected_anomalies")
    )
)

display(spatial_zero_summary)

evaluated_spatial_groups,detected_anomalies
972,6


In [0]:
# ============================================================
# REVIEW ZERO-EGG RATE ANOMALIES
# ============================================================

# Display the spatial groups classified as anomalous
# according to the persistent zero-egg rate.
display(
    df_spatial_zero_test
    .filter(F.col("is_zero_rate_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "number_of_days",
        "median_spatial_deviation_pct",
        "median_zero_egg_rate_pct",
        "is_zero_rate_anomaly"
    )
    .orderBy(
        F.desc("median_zero_egg_rate_pct")
    )
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,median_spatial_deviation_pct,median_zero_egg_rate_pct,is_zero_rate_anomaly
farm_02,2,4,1,BACK,451,5,-100.0,100.0,true
farm_02,2,4,1,FRONT,451,5,-100.0,100.0,true
farm_02,2,4,2,BACK,451,5,-100.0,100.0,true
farm_02,2,4,2,FRONT,451,5,-100.0,100.0,true
farm_03,1,4,3,FRONT,301,5,-18.67,24.0,true
farm_03,2,3,1,BACK,251,5,0.0,20.0,true


In [0]:
# ============================================================
# COMBINE SPATIAL ANOMALY CRITERIA
# ============================================================

# Classify a spatial group as anomalous only when it shows
# both an unusually low production level and an unusually
# high persistent zero-egg rate.
df_spatial_detected = (
    df_spatial_persistent
    .withColumn(
        "is_spatial_anomaly",
        (
            F.col("median_spatial_deviation_pct")
            < F.lit(lower_spatial_threshold)
        )
        &
        (
            F.col("median_zero_egg_rate_pct")
            > F.lit(upper_zero_threshold)
        )
    )
)

In [0]:
# ============================================================
# REVIEW DETECTED SPATIAL ANOMALIES
# ============================================================

# Display only the spatial groups satisfying both
# automatically derived anomaly criteria.
display(
    df_spatial_detected
    .filter(F.col("is_spatial_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "number_of_days",
        "median_spatial_deviation_pct",
        "median_zero_egg_rate_pct",
        "is_spatial_anomaly"
    )
    .orderBy(
        "median_spatial_deviation_pct"
    )
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,median_spatial_deviation_pct,median_zero_egg_rate_pct,is_spatial_anomaly
farm_02,2,4,1,BACK,451,5,-100.0,100.0,true
farm_02,2,4,1,FRONT,451,5,-100.0,100.0,true
farm_02,2,4,2,BACK,451,5,-100.0,100.0,true
farm_02,2,4,2,FRONT,451,5,-100.0,100.0,true
farm_03,1,4,3,FRONT,301,5,-18.67,24.0,true


In [0]:
# ============================================================
# EVALUATE DAILY PERSISTENCE OF SPATIAL CANDIDATES
# ============================================================

# Select the spatial groups classified as persistent
# anomaly candidates by the combined IQR criteria.
spatial_candidates = (
    df_spatial_detected
    .filter(F.col("is_spatial_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group"
    )
)

# Join the candidates with the original daily spatial data
# to recover their behaviour on each individual day.
df_candidate_daily = (
    df_spatial_detect
    .join(
        spatial_candidates,
        on=[
            "farm_id",
            "house_number",
            "battery_number",
            "level",
            "side",
            "position_group"
        ],
        how="inner"
    )
)

# Display the daily behaviour of each candidate zone.
display(
    df_candidate_daily
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "capture_date",
        "mean_egg_count",
        "battery_median_egg_count",
        "zero_egg_rate_pct",
        "spatial_deviation_pct"
    )
    .orderBy(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "capture_date"
    )
)

farm_id,house_number,battery_number,level,side,position_group,capture_date,mean_egg_count,battery_median_egg_count,zero_egg_rate_pct,spatial_deviation_pct
farm_02,2,4,1,BACK,451,2026-08-01,0.0,0.98,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-02,0.0,0.94,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-03,0.0,1.0,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-04,0.0,0.98,100.0,-100.0
farm_02,2,4,1,BACK,451,2026-08-05,0.0,0.98,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-01,0.0,0.98,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-02,0.0,0.94,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-03,0.0,1.0,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-04,0.0,0.98,100.0,-100.0
farm_02,2,4,1,FRONT,451,2026-08-05,0.0,0.98,100.0,-100.0


In [0]:
# ============================================================
# EVALUATE DAILY SPATIAL ANOMALY PERSISTENCE
# ============================================================

# Classify each daily spatial observation using the same
# automatically derived IQR thresholds used previously.
df_spatial_daily_flags = (
    df_spatial_detect
    .withColumn(
        "is_daily_spatial_anomaly",
        (
            F.col("spatial_deviation_pct")
            < F.lit(lower_spatial_threshold)
        )
        &
        (
            F.col("zero_egg_rate_pct")
            > F.lit(upper_zero_threshold)
        )
    )
)

# Count how many days each spatial group behaves anomalously.
df_spatial_persistence = (
    df_spatial_daily_flags
    .groupBy(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group"
    )
    .agg(
        F.countDistinct("capture_date").alias("number_of_days"),

        F.sum(
            F.col("is_daily_spatial_anomaly").cast("int")
        ).alias("anomalous_days")
    )
    .withColumn(
        "anomalous_days_pct",
        F.round(
            F.col("anomalous_days")
            / F.col("number_of_days")
            * 100,
            2
        )
    )
)

In [0]:
# ============================================================
# REVIEW SPATIAL ANOMALY PERSISTENCE
# ============================================================

# Display spatial groups ordered by the percentage of days
# in which anomalous behaviour was detected.
display(
    df_spatial_persistence
    .filter(F.col("anomalous_days") > 0)
    .orderBy(
        F.desc("anomalous_days_pct"),
        F.desc("anomalous_days")
    )
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,anomalous_days,anomalous_days_pct
farm_02,2,4,1,FRONT,451,5,5,100.0
farm_02,2,4,2,FRONT,451,5,5,100.0
farm_02,2,4,1,BACK,451,5,5,100.0
farm_02,2,4,2,BACK,451,5,5,100.0
farm_03,1,4,3,FRONT,301,5,3,60.0
farm_03,2,3,3,BACK,51,5,2,40.0
farm_03,2,1,1,BACK,151,5,2,40.0
farm_03,1,4,3,BACK,1,5,2,40.0
farm_03,2,3,1,BACK,251,5,2,40.0
farm_03,2,2,3,FRONT,251,5,2,40.0


In [0]:
# ============================================================
# IDENTIFY PERSISTENT SPATIAL ANOMALIES
# ============================================================

# Calculate whether the anomalous behaviour was observed
# throughout the complete period available for each spatial group.
#
# In this prototype, a spatial anomaly is considered persistent
# when it is detected on every available day.
df_spatial_final = (
    df_spatial_persistence
    .withColumn(
        "is_persistent_spatial_anomaly",
        F.col("anomalous_days") == F.col("number_of_days")
    )
)

In [0]:
# ============================================================
# REVIEW FINAL SPATIAL ANOMALIES
# ============================================================

# Display the spatial groups classified as persistent anomalies.
display(
    df_spatial_final
    .filter(F.col("is_persistent_spatial_anomaly"))
    .orderBy(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group"
    )
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,anomalous_days,anomalous_days_pct,is_persistent_spatial_anomaly
farm_02,2,4,1,BACK,451,5,5,100.0,true
farm_02,2,4,1,FRONT,451,5,5,100.0,true
farm_02,2,4,2,BACK,451,5,5,100.0,true
farm_02,2,4,2,FRONT,451,5,5,100.0,true


In [0]:
# ============================================================
# PREPARE TEMPORAL ANOMALIES
# ============================================================

# Transform the detected temporal anomalies into a common
# structure that can later be combined with spatial anomalies.
df_temporal_anomalies = (
    df_temporal_detected
    .filter(F.col("is_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "capture_date",
        F.lit("TEMPORAL").alias("anomaly_type"),
        F.col("deviation_pct").alias("deviation_pct"),
        F.col("mean_egg_count"),
        F.col("baseline_median"),
        F.lit(None).cast("int").alias("level"),
        F.lit(None).cast("string").alias("side"),
        F.lit(None).cast("int").alias("position_group"),
        F.lit(None).cast("double").alias("persistence_pct")
    )
)

In [0]:
# ============================================================
# PREPARE SPATIAL ANOMALIES
# ============================================================

# Transform the persistent spatial anomalies into the same
# structure used for temporal anomalies.
df_spatial_anomalies = (
    df_spatial_final
    .filter(F.col("is_persistent_spatial_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        F.lit(None).cast("date").alias("capture_date"),
        F.lit("SPATIAL").alias("anomaly_type"),
        F.lit(None).cast("double").alias("deviation_pct"),
        F.lit(None).cast("double").alias("mean_egg_count"),
        F.lit(None).cast("double").alias("baseline_median"),
        "level",
        "side",
        "position_group",
        F.col("anomalous_days_pct").alias("persistence_pct")
    )
)

In [0]:
# ============================================================
# COMBINE DETECTED ANOMALIES
# ============================================================

# Combine temporal and spatial anomalies into a single
# analytical dataset with a common schema.
df_detected_anomalies = (
    df_temporal_anomalies
    .unionByName(df_spatial_anomalies)
)

# Review the final anomaly dataset before storing it.
display(
    df_detected_anomalies
    .orderBy(
        "anomaly_type",
        "farm_id",
        "house_number",
        "battery_number"
    )
)

print(
    f"Total detected anomalies: "
    f"{df_detected_anomalies.count()}"
)

farm_id,house_number,battery_number,capture_date,anomaly_type,deviation_pct,mean_egg_count,baseline_median,level,side,position_group,persistence_pct
farm_02,2,4,null,SPATIAL,null,null,null,1,FRONT,451,100.0
farm_02,2,4,null,SPATIAL,null,null,null,2,FRONT,451,100.0
farm_02,2,4,null,SPATIAL,null,null,null,1,BACK,451,100.0
farm_02,2,4,null,SPATIAL,null,null,null,2,BACK,451,100.0
farm_03,1,1,2026-08-04,TEMPORAL,-47.56,0.515,0.982,null,null,null,null
farm_03,1,2,2026-08-04,TEMPORAL,-45.3,0.53,0.969,null,null,null,null
farm_03,1,3,2026-08-04,TEMPORAL,-48.94,0.507,0.993,null,null,null,null
farm_03,1,4,2026-08-04,TEMPORAL,-46.92,0.525,0.989,null,null,null,null
farm_03,2,1,2026-08-04,TEMPORAL,-47.79,0.509,0.975,null,null,null,null
farm_03,2,2,2026-08-04,TEMPORAL,-45.41,0.535,0.98,null,null,null,null


Total detected anomalies: 11


In [0]:
# ============================================================
# SAVE DETECTED ANOMALIES TO GOLD
# ============================================================

# Store the final anomaly dataset in Delta format.
# The dataset contains both temporal and persistent
# spatial anomalies detected automatically.
(
    df_detected_anomalies
    .write
    .format("delta")
    .mode("overwrite")
    .save(gold_anomalies_path)
)

print(
    f"Detected anomalies saved successfully to: "
    f"{gold_anomalies_path}"
)

Detected anomalies saved successfully to: abfss://lakehouse@mastermmf001sta.dfs.core.windows.net/gold/detected_anomalies


In [0]:
# ============================================================
# VALIDATE STORED ANOMALY DATASET
# ============================================================

# Read the stored Delta dataset to verify that the
# anomaly detection results were written correctly.
df_anomalies_stored = (
    spark.read
    .format("delta")
    .load(gold_anomalies_path)
)

# Count the stored anomalies by type.
display(
    df_anomalies_stored
    .groupBy("anomaly_type")
    .count()
    .orderBy("anomaly_type")
)

print(
    f"Total stored anomalies: "
    f"{df_anomalies_stored.count()}"
)

anomaly_type,count
SPATIAL,4
TEMPORAL,7


Total stored anomalies: 11


In [0]:
# ============================================================
# DISPLAY PERSISTENT SPATIAL ANOMALIES
# ============================================================

# Display only the relevant variables describing the
# persistent spatial anomalies detected in the dataset.
display(
    df_spatial_final
    .filter(F.col("is_persistent_spatial_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side",
        "position_group",
        "number_of_days",
        "anomalous_days",
        "anomalous_days_pct"
    )
    .orderBy(
        "farm_id",
        "house_number",
        "battery_number",
        "level",
        "side"
    )
)

farm_id,house_number,battery_number,level,side,position_group,number_of_days,anomalous_days,anomalous_days_pct
farm_02,2,4,1,BACK,451,5,5,100.0
farm_02,2,4,1,FRONT,451,5,5,100.0
farm_02,2,4,2,BACK,451,5,5,100.0
farm_02,2,4,2,FRONT,451,5,5,100.0


In [0]:
# ============================================================
# DISPLAY TEMPORAL ANOMALIES
# ============================================================

# Display only the relevant variables describing the
# temporal anomalies detected in the dataset.
display(
    df_temporal_detected
    .filter(F.col("is_anomaly"))
    .select(
        "farm_id",
        "house_number",
        "battery_number",
        "capture_date",
        "mean_egg_count",
        "baseline_median",
        "deviation_pct"
    )
    .orderBy(
        "farm_id",
        "house_number",
        "battery_number"
    )
)

farm_id,house_number,battery_number,capture_date,mean_egg_count,baseline_median,deviation_pct
farm_03,1,1,2026-08-04,0.515,0.982,-47.56
farm_03,1,2,2026-08-04,0.53,0.969,-45.3
farm_03,1,3,2026-08-04,0.507,0.993,-48.94
farm_03,1,4,2026-08-04,0.525,0.989,-46.92
farm_03,2,1,2026-08-04,0.509,0.975,-47.79
farm_03,2,2,2026-08-04,0.535,0.98,-45.41
farm_03,2,3,2026-08-04,0.514,0.975,-47.28
